In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import os
import time

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import joblib

PROJECT_ROOT = Path.cwd()
KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
OUTPUT_DIR = KAGGLE_WORKING if KAGGLE_WORKING.exists() else PROJECT_ROOT
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR_CANDIDATES = [
    Path(os.environ['CIC_IDS_DATA_DIR']) if os.environ.get('CIC_IDS_DATA_DIR') else None,
    PROJECT_ROOT,
    PROJECT_ROOT / 'datasets',
    KAGGLE_INPUT,
    KAGGLE_INPUT / 'compressed-cic-2018',
    KAGGLE_WORKING,
]
DATA_DIR_CANDIDATES = [p for p in DATA_DIR_CANDIDATES if p is not None]


def find_data_file(filename: str) -> Path:
    for base in DATA_DIR_CANDIDATES:
        candidate = base / filename
        if candidate.exists():
            return candidate
    searched = '\n'.join(str(p / filename) for p in DATA_DIR_CANDIDATES)
    raise FileNotFoundError(f"Could not find {filename}. Searched:\n{searched}")


def artifact_path(filename: str) -> Path:
    return OUTPUT_DIR / filename


def hierarchical_f1(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    f1_binary = f1_score((y_true > 0), (y_pred > 0), average='macro', zero_division=0)

    attack_mask = y_true > 0
    f1_attack = f1_score(
        y_true[attack_mask],
        y_pred[attack_mask],
        average='macro',
        zero_division=0
    ) if attack_mask.any() else 0.0

    return min(f1_binary, f1_attack)


def add_binary_metrics(y_true, y_pred):
    y_true_bin = (np.asarray(y_true) != 0).astype(int)
    y_pred_bin = (np.asarray(y_pred) != 0).astype(int)
    return {
        'multiclass_accuracy': accuracy_score(y_true, y_pred),
        'hierarchical_f1': hierarchical_f1(y_true, y_pred),
        'binary_precision': precision_score(y_true_bin, y_pred_bin, zero_division=0),
        'binary_recall': recall_score(y_true_bin, y_pred_bin, zero_division=0),
        'binary_f1': f1_score(y_true_bin, y_pred_bin, zero_division=0),
    }

/mnt/c/users/vihaa/OneDrive/Documents/IPD-IDS/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load compressed VAE data
train_compressed_path = find_data_file('df_train.csv')
test_compressed_path = find_data_file('df_test.csv')
df_train = pd.read_csv(train_compressed_path)
df_test = pd.read_csv(test_compressed_path)

print(f'Train path: {train_compressed_path}')
print(f'Test path: {test_compressed_path}')
print(f'Train shape: {df_train.shape}')
print(f'Test shape: {df_test.shape}')
print(f"\nTrain label distribution:\n{df_train['label'].value_counts()}")
print(f"\nTest label distribution:\n{df_test['label'].value_counts()}")

Train path: /mnt/c/users/vihaa/OneDrive/Documents/IPD-IDS/df_train.csv
Test path: /mnt/c/users/vihaa/OneDrive/Documents/IPD-IDS/df_test.csv
Train shape: (2049029, 8)
Test shape: (1048575, 8)

Train label distribution:
label
0    1663709
4     187587
3     145241
1      41508
2      10984
Name: count, dtype: int64

Test label distribution:
label
8    686012
0    360833
9      1730
Name: count, dtype: int64


In [3]:
# Prepare features and labels
feature_cols = ['latent_0', 'latent_1', 'latent_2', 'latent_3', 'latent_4', 'recon_loss', 'kld_loss']

X_train_full = df_train[feature_cols].values
y_train_full = df_train['label'].values

X_test = df_test[feature_cols].values
y_test = df_test['label'].values

# Compute inverse class weights
sample_weights_full = compute_sample_weight(class_weight='balanced', y=y_train_full)

print(f"X_train_full: {X_train_full.shape}, X_test: {X_test.shape}")
print(f"\nClass distribution in training:")
unique, counts = np.unique(y_train_full, return_counts=True)
for c, cnt in zip(unique, counts):
    print(f"  Class {c}: {cnt} samples, weight: {len(y_train_full) / (len(unique) * cnt):.4f}")

X_train_full: (2049029, 7), X_test: (1048575, 7)

Class distribution in training:
  Class 0: 1663709 samples, weight: 0.2463
  Class 1: 41508 samples, weight: 9.8729
  Class 2: 10984 samples, weight: 37.3093
  Class 3: 145241 samples, weight: 2.8216
  Class 4: 187587 samples, weight: 2.1846


# Optuna Hyperparameter Tuning

In [ ]:
def objective(trial):
    params = {
        'objective': 'multi:softmax',
        'num_class': len(np.unique(y_train_full)),
        'tree_method': 'hist',
        'device': 'cuda',
        'sampling_method': 'gradient_based',
        'max_bin': 511,
        'eval_metric': 'mlogloss',
        'random_state': 42,
        'n_estimators': trial.suggest_int('n_estimators', 50, 1000),
        'max_depth': trial.suggest_int('max_depth', 2, 20),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.5, log=True),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.4, 1.0),
        'colsample_bynode': trial.suggest_float('colsample_bynode', 0.4, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-10, 100.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-10, 100.0, log=True),
        'gamma': trial.suggest_float('gamma', 1e-10, 10.0, log=True),
        'max_delta_step': trial.suggest_int('max_delta_step', 0, 10),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 0.5, 5.0),
    }
    
    # Stratified K-Fold cross-validation
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    f1_scores = []
    
    for train_idx, val_idx in skf.split(X_train_full, y_train_full):
        X_train_fold = X_train_full[train_idx]
        y_train_fold = y_train_full[train_idx]
        X_val_fold = X_train_full[val_idx]
        y_val_fold = y_train_full[val_idx]
        sw_train_fold = sample_weights_full[train_idx]
        
        model = xgb.XGBClassifier(**params)
        model.fit(
            X_train_fold, y_train_fold,
            sample_weight=sw_train_fold,
            eval_set=[(X_val_fold, y_val_fold)],
            verbose=False
        )
        
        y_pred = model.predict(X_val_fold)
        f1_scores.append(hierarchical_f1(y_val_fold, y_pred))
    
    return np.mean(f1_scores)

# Run Optuna study
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, n_jobs=1, show_progress_bar=True)

print(f"\nBest trial hierarchical F1: {study.best_trial.value:.4f}")
print(f"Best params: {study.best_trial.params}")

[I 2026-05-17 12:28:31,548] A new study created in memory with name: no-name-145fee3e-e53f-4a0b-9022-5877b3f3119e
Best trial: 0. Best value: 0.7429:   5%|▌         | 1/20 [15:21<4:51:57, 921.96s/it]

[I 2026-05-17 12:43:53,505] Trial 0 finished with value: 0.7429001009962808 and parameters: {'n_estimators': 463, 'max_depth': 18, 'learning_rate': 0.15872177199786774, 'subsample': 0.8315052934384661, 'colsample_bytree': 0.5279420249740103, 'colsample_bylevel': 0.831418590314907, 'colsample_bynode': 0.913430216216112, 'min_child_weight': 9, 'reg_alpha': 3.947300333273082e-08, 'reg_lambda': 6.944407921061188e-10, 'gamma': 1.872037383580881e-09, 'max_delta_step': 1, 'scale_pos_weight': 3.848308403271714}. Best is trial 0 with value: 0.7429001009962808.


Best trial: 1. Best value: 0.748008:  10%|█         | 2/20 [29:39<4:25:09, 883.88s/it]

[I 2026-05-17 12:58:10,731] Trial 1 finished with value: 0.7480081972560338 and parameters: {'n_estimators': 995, 'max_depth': 16, 'learning_rate': 0.15486111866148233, 'subsample': 0.6308453126009987, 'colsample_bytree': 0.9876076727993851, 'colsample_bylevel': 0.976062131449663, 'colsample_bynode': 0.5998809083537082, 'min_child_weight': 4, 'reg_alpha': 0.040789208100603026, 'reg_lambda': 2.6178961606113747e-09, 'gamma': 0.002744233239965191, 'max_delta_step': 6, 'scale_pos_weight': 1.0846946721007662}. Best is trial 1 with value: 0.7480081972560338.


Best trial: 1. Best value: 0.748008:  15%|█▌        | 3/20 [35:38<3:02:34, 644.37s/it]

...

[I 2026-05-17 13:54:15,291] Trial 5 finished with value: 0.7239446829408894 and parameters: {'n_estimators': 583, 'max_depth': 2, 'learning_rate': 0.46826614217121376, 'subsample': 0.5365826181735335, 'colsample_bytree': 0.7925538205595506, 'colsample_bylevel': 0.5930810368879336, 'colsample_bynode': 0.8072163674308521, 'min_child_weight': 17, 'reg_alpha': 1.328379745162813e-07, 'reg_lambda': 9.466769381638587e-07, 'gamma': 9.118887580820686e-06, 'max_delta_step': 1, 'scale_pos_weight': 4.586932613572148}. Best is trial 1 with value: 0.7480081972560338.


Best trial: 1. Best value: 0.748008:  35%|███▌      | 7/20 [2:10:18<5:08:54, 1425.72s/it]

[I 2026-05-17 14:38:49,617] Trial 6 finished with value: 0.7478072244828863 and parameters: {'n_estimators': 609, 'max_depth': 18, 'learning_rate': 0.01865937188837038, 'subsample': 0.5827783497302794, 'colsample_bytree': 0.7991190091720699, 'colsample_bylevel': 0.5868457351249152, 'colsample_bynode': 0.6100088966388162, 'min_child_weight': 3, 'reg_alpha': 2.911960399254679, 'reg_lambda': 1.1844904937626107e-10, 'gamma': 0.09005138895811736, 'max_delta_step': 0, 'scale_pos_weight': 0.9098866419765497}. Best is trial 1 with value: 0.7480081972560338.


Best trial: 1. Best value: 0.748008:  35%|███▌      | 7/20 [2:13:54<4:08:41, 1147.79s/it]


[W 2026-05-17 14:42:26,047] Trial 7 failed with parameters: {'n_estimators': 813, 'max_depth': 2, 'learning_rate': 0.013899521945451984, 'subsample': 0.4644882951093962, 'colsample_bytree': 0.8655560295055997, 'colsample_bylevel': 0.9632338477584313, 'colsample_bynode': 0.6286837259552833, 'min_child_weight': 17, 'reg_alpha': 0.04070880349263353, 'reg_lambda': 7.005027797566428, 'gamma': 0.24317344080064981, 'max_delta_step': 6, 'scale_pos_weight': 3.589227920541102} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/mnt/c/users/vihaa/OneDrive/Documents/IPD-IDS/.venv/lib/python3.12/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_90912/240488208.py", line 38, in objective
    model.fit(
  File "/mnt/c/users/vihaa/OneDrive/Documents/IPD-IDS/.venv/lib/python3.12/site-packages/xgboost/core.py", line 751, in inner_f
    return func(**kwar

# Train Final Model with Best Params

In [ ]:
# Train final model on full training data with best params
best_params = study.best_trial.params
best_params.update({
    'objective': 'multi:softmax',
    'num_class': len(np.unique(y_train_full)),
    'tree_method': 'hist',
    'device': 'cuda',
    'sampling_method': 'gradient_based',
    'max_bin': 512,
    'predictor': 'gpu_predictor',
    'eval_metric': 'mlogloss',
    'random_state': 42,
})

final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train_full, y_train_full, sample_weight=sample_weights_full, verbose=True)

print("Final model trained on full training data with class weights!")

Final model trained on full training data with class weights!


# Final Evaluation on Test Set

In [ ]:
# Evaluate on test set
y_pred_test = final_model.predict(X_test)

# Compute sample weights for test set (multiclass)
sample_weights_test = compute_sample_weight(class_weight='balanced', y=y_test)

# Create binary labels for test: 0 = benign, 1 = attack (any non-0)
y_test_binary = (y_test != 0).astype(int)
y_pred_binary = (y_pred_test != 0).astype(int)

# Compute binary class weights (inverse frequency for 0 vs 1)
sample_weights_binary = compute_sample_weight(class_weight='balanced', y=y_test_binary)

# Custom weighted binary accuracy
def weighted_binary_accuracy(y_true_bin, y_pred_bin, sample_weights):
    correct_mask = y_true_bin == y_pred_bin
    
    benign_mask = y_true_bin == 0
    attack_mask = y_true_bin == 1
    
    benign_correct_weighted = np.sum(sample_weights[benign_mask & correct_mask])
    benign_total_weighted = np.sum(sample_weights[benign_mask])
    
    attack_correct_weighted = np.sum(sample_weights[attack_mask & correct_mask])
    attack_total_weighted = np.sum(sample_weights[attack_mask])
    
    total_correct_weighted = benign_correct_weighted + attack_correct_weighted
    total_weighted = np.sum(sample_weights)
    
    print(f"\nWeighted Binary Evaluation (Attack vs Benign):")
    print(f"  Benign (0) weighted acc: {benign_correct_weighted:.2f}/{benign_total_weighted:.2f} = {benign_correct_weighted/benign_total_weighted:.4f}")
    print(f"  Attack (1) weighted acc: {attack_correct_weighted:.2f}/{attack_total_weighted:.2f} = {attack_correct_weighted/attack_total_weighted:.4f}")
    
    return total_correct_weighted / total_weighted

print("=" * 60)
print("FINAL TEST SET EVALUATION")
print("=" * 60)

# Binary class distribution
n_benign = np.sum(y_test_binary == 0)
n_attack = np.sum(y_test_binary == 1)
print(f"\nBinary test distribution: Benign={n_benign}, Attack={n_attack}")
print(f"Binary weights: Benign={len(y_test_binary)/(2*n_benign):.4f}, Attack={len(y_test_binary)/(2*n_attack):.4f}")

print(f"\nAccuracy (multiclass):  {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Accuracy (multiclass weighted): {accuracy_score(y_test, y_pred_test, sample_weight=sample_weights_test):.4f}")
print(f"Accuracy (binary): {accuracy_score(y_test_binary, y_pred_binary):.4f}")
weighted_bin_acc = weighted_binary_accuracy(y_test_binary, y_pred_binary, sample_weights_binary)
print(f"Accuracy (binary weighted): {weighted_bin_acc:.4f}")

print(f"\nHierarchical F1: {hierarchical_f1(y_test, y_pred_test):.4f}")
print(f"Precision (macro): {precision_score(y_test, y_pred_test, average='macro'):.4f}")
print(f"Recall (macro): {recall_score(y_test, y_pred_test, average='macro'):.4f}")

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT (Binary: Attack vs Benign)")
print("=" * 60)
print(classification_report(y_test_binary, y_pred_binary, target_names=['Benign', 'Attack']))

FINAL TEST SET EVALUATION

Binary test distribution: Benign=45283, Attack=55718
Binary weights: Benign=1.1152, Attack=0.9064

Accuracy (multiclass):  0.4474
Accuracy (multiclass weighted): 0.3326
Accuracy (binary): 0.9764

Weighted Binary Evaluation (Attack vs Benign):
  Benign (0) weighted acc: 50390.09/50500.50 = 0.9978
  Attack (1) weighted acc: 48428.56/50500.50 = 0.9590
Accuracy (binary weighted): 0.9784

F1 (weighted): 0.4368
Precision (weighted): 0.4268
Recall (weighted): 0.4474

CLASSIFICATION REPORT (Multiclass)
              precision    recall  f1-score   support

           0       0.95      1.00      0.97     45283
           1       0.00      0.00      0.00         0
           2       0.00      0.00      0.00         0
           3       0.00      0.00      0.00         0
           4       0.00      0.00      0.00         0
           8       0.00      0.00      0.00     53996
           9       0.00      0.00      0.00      1722

    accuracy                           

In [ ]:
# Save the model
model_path = artifact_path('xgboost_model.json')
final_model.save_model(str(model_path))
print(f'Model saved to {model_path}')

Model saved to /kaggle/working/xgboost_model.json


In [ ]:
# Benchmark helpers: tuning functions and shared state
import gc
import time

latent_cols = [c for c in feature_cols if c.startswith('latent_')]
benchmark_rows = []
benchmark_models = {}
benchmark_predictions = {}


def build_xgb_params(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }


def tune_feature_set_xgb(X_train, y_train):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    def objective(trial):
        params = build_xgb_params(trial)
        params.update({
            'objective': 'multi:softmax',
            'num_class': len(np.unique(y_train)),
            'tree_method': 'hist',
            'device': 'cuda',
    'sampling_method': 'gradient_based',
    'max_bin': 512,
    'predictor': 'gpu_predictor',
            'sampling_method': 'gradient_based',
            'max_bin': 512,
            'predictor': 'gpu_predictor',
        'sampling_method': 'gradient_based',
        'max_bin': 512,
        'predictor': 'gpu_predictor',
            'eval_metric': 'mlogloss',
            'random_state': 42,
        })
        fold_scores = []

        for train_idx, val_idx in skf.split(X_train, y_train):
            X_fold_train = X_train[train_idx]
            y_fold_train = y_train[train_idx]
            X_fold_val = X_train[val_idx]
            y_fold_val = y_train[val_idx]
            fold_weights = compute_sample_weight(class_weight='balanced', y=y_fold_train)

            model = xgb.XGBClassifier(**params)
            model.fit(
                X_fold_train, y_fold_train,
                sample_weight=fold_weights,
                eval_set=[(X_fold_val, y_fold_val)],
                verbose=False
            )
            y_fold_pred = model.predict(X_fold_val)
            fold_scores.append(hierarchical_f1(y_fold_val, y_fold_pred))

        return float(np.mean(fold_scores))

    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=20, n_jobs=1, show_progress_bar=True, catch=(Exception,))
    return study


def run_feature_set_benchmark_xgb(name, X_train, y_train, X_test, y_test):
    print("=" * 60)
    print(f"BENCHMARK: {name}")
    print("=" * 60)
    print(f"Training shape: {X_train.shape}")
    print(f"Test shape: {X_test.shape}")

    gc.collect()
    tuning_start = time.time()
    study = tune_feature_set_xgb(X_train, y_train)
    tuning_seconds = time.time() - tuning_start

    best_params = study.best_trial.params.copy()
    best_params.update({
        'objective': 'multi:softmax',
        'num_class': len(np.unique(y_train)),
        'tree_method': 'hist',
        'device': 'cuda',
    'sampling_method': 'gradient_based',
    'max_bin': 512,
    'predictor': 'gpu_predictor',
        'sampling_method': 'gradient_based',
        'max_bin': 512,
        'predictor': 'gpu_predictor',
        'eval_metric': 'mlogloss',
        'random_state': 42,
    })

    model = xgb.XGBClassifier(**best_params)
    fit_start = time.time()
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
    model.fit(X_train, y_train, sample_weight=sample_weights, verbose=False)
    fit_seconds = time.time() - fit_start

    y_pred = model.predict(X_test)
    y_test_binary = (np.asarray(y_test) != 0).astype(int)
    y_pred_binary = (np.asarray(y_pred) != 0).astype(int)
    sample_weights_binary = compute_sample_weight(class_weight='balanced', y=y_test_binary)
    metrics = add_binary_metrics(y_test, y_pred)

    print("\n" + "=" * 60)
    print("FINAL TEST SET EVALUATION")
    print("=" * 60)
    print(f"\nBinary test distribution: Benign={np.sum(y_test_binary == 0)}, Attack={np.sum(y_test_binary == 1)}")
    print(f"Binary weights: Benign={len(y_test_binary)/(2*np.sum(y_test_binary == 0)):.4f}, Attack={len(y_test_binary)/(2*np.sum(y_test_binary == 1)):.4f}")
    print(f"\nAccuracy (multiclass):  {accuracy_score(y_test, y_pred):.4f}")
    print(f"Accuracy (multiclass weighted): {accuracy_score(y_test, y_pred, sample_weight=compute_sample_weight(class_weight='balanced', y=y_test)):.4f}")
    print(f"Accuracy (binary): {accuracy_score(y_test_binary, y_pred_binary):.4f}")
    print("\nWeighted Binary Evaluation (Attack vs Benign):")
    correct_mask = y_test_binary == y_pred_binary
    benign_mask = y_test_binary == 0
    attack_mask = y_test_binary == 1
    benign_correct_weighted = np.sum(sample_weights_binary[benign_mask & correct_mask])
    benign_total_weighted = np.sum(sample_weights_binary[benign_mask])
    attack_correct_weighted = np.sum(sample_weights_binary[attack_mask & correct_mask])
    attack_total_weighted = np.sum(sample_weights_binary[attack_mask])
    total_correct_weighted = benign_correct_weighted + attack_correct_weighted
    total_weighted = np.sum(sample_weights_binary)
    print(f"  Benign (0) weighted acc: {benign_correct_weighted:.2f}/{benign_total_weighted:.2f} = {benign_correct_weighted/benign_total_weighted:.4f}")
    print(f"  Attack (1) weighted acc: {attack_correct_weighted:.2f}/{attack_total_weighted:.2f} = {attack_correct_weighted/attack_total_weighted:.4f}")
    print(f"Accuracy (binary weighted): {total_correct_weighted / total_weighted:.4f}")
    print(f"\nHierarchical F1: {hierarchical_f1(y_test, y_pred):.4f}")
    print(f"Precision (macro): {precision_score(y_test, y_pred, average='macro'):.4f}")
    print(f"Recall (macro): {recall_score(y_test, y_pred, average='macro'):.4f}")

    print("\n" + "=" * 60)
    print("CLASSIFICATION REPORT (Binary: Attack vs Benign)")
    print("=" * 60)
    print(classification_report(y_test_binary, y_pred_binary, target_names=['Benign', 'Attack']))

    row = {
        'feature_set': name,
        'tuning_seconds': tuning_seconds,
        'fit_seconds': fit_seconds,
        'train_seconds': tuning_seconds + fit_seconds,
        'best_trial_hierarchical_f1': study.best_trial.value,
        'n_trials': 20,
    }
    row.update(metrics)

    benchmark_rows.append(row)
    benchmark_models[name] = model
    benchmark_predictions[name] = y_pred

    return row


def add_pretrained_model_benchmark_xgb(name, model_obj, X_test, y_test):
    """Add results from an already-trained model (like final_model) to benchmark."""
    print("=" * 60)
    print(f"BENCHMARK: {name} (using pre-trained final model)")
    print("=" * 60)
    print(f"Test shape: {X_test.shape}")

    y_pred = model_obj.predict(X_test)
    y_test_binary = (np.asarray(y_test) != 0).astype(int)
    y_pred_binary = (np.asarray(y_pred) != 0).astype(int)
    sample_weights_binary = compute_sample_weight(class_weight='balanced', y=y_test_binary)
    metrics = add_binary_metrics(y_test, y_pred)

    print("\n" + "=" * 60)
    print("FINAL TEST SET EVALUATION")
    print("=" * 60)
    print(f"\nBinary test distribution: Benign={np.sum(y_test_binary == 0)}, Attack={np.sum(y_test_binary == 1)}")
    print(f"Binary weights: Benign={len(y_test_binary)/(2*np.sum(y_test_binary == 0)):.4f}, Attack={len(y_test_binary)/(2*np.sum(y_test_binary == 1)):.4f}")
    print(f"\nAccuracy (multiclass):  {accuracy_score(y_test, y_pred):.4f}")
    print(f"Accuracy (multiclass weighted): {accuracy_score(y_test, y_pred, sample_weight=compute_sample_weight(class_weight='balanced', y=y_test)):.4f}")
    print(f"Accuracy (binary): {accuracy_score(y_test_binary, y_pred_binary):.4f}")
    print("\nWeighted Binary Evaluation (Attack vs Benign):")
    correct_mask = y_test_binary == y_pred_binary
    benign_mask = y_test_binary == 0
    attack_mask = y_test_binary == 1
    benign_correct_weighted = np.sum(sample_weights_binary[benign_mask & correct_mask])
    benign_total_weighted = np.sum(sample_weights_binary[benign_mask])
    attack_correct_weighted = np.sum(sample_weights_binary[attack_mask & correct_mask])
    attack_total_weighted = np.sum(sample_weights_binary[attack_mask])
    total_correct_weighted = benign_correct_weighted + attack_correct_weighted
    total_weighted = np.sum(sample_weights_binary)
    print(f"  Benign (0) weighted acc: {benign_correct_weighted:.2f}/{benign_total_weighted:.2f} = {benign_correct_weighted/benign_total_weighted:.4f}")
    print(f"  Attack (1) weighted acc: {attack_correct_weighted:.2f}/{attack_total_weighted:.2f} = {attack_correct_weighted/attack_total_weighted:.4f}")
    print(f"Accuracy (binary weighted): {total_correct_weighted / total_weighted:.4f}")
    print(f"\nHierarchical F1: {hierarchical_f1(y_test, y_pred):.4f}")
    print(f"Precision (macro): {precision_score(y_test, y_pred, average='macro'):.4f}")
    print(f"Recall (macro): {recall_score(y_test, y_pred, average='macro'):.4f}")

    print("\n" + "=" * 60)
    print("CLASSIFICATION REPORT (Binary: Attack vs Benign)")
    print("=" * 60)
    print(classification_report(y_test_binary, y_pred_binary, target_names=['Benign', 'Attack']))

    row = {
        'feature_set': name,
        'tuning_seconds': 0,
        'fit_seconds': 0,
        'train_seconds': 0,
        'best_trial_hierarchical_f1': 0,
        'n_trials': 0,
    }
    row.update(metrics)

    benchmark_rows.append(row)
    benchmark_models[name] = model_obj
    benchmark_predictions[name] = y_pred

    return row

In [ ]:
compressed_loss_result = add_pretrained_model_benchmark_xgb(
    'compressed + recon/kld loss',
    final_model,
    df_test[feature_cols].values,
    y_test,
)


In [ ]:
compressed_latent_result = run_feature_set_benchmark_xgb(
    'compressed latent only',
    df_train[latent_cols].values,
    y_train_full,
    df_test[latent_cols].values,
    y_test,
)


In [ ]:
full_original_result = run_feature_set_benchmark_xgb(
    'full original features',
    df_train.drop(columns=['label']).values,
    df_train['label'].values,
    df_test.drop(columns=['label']).values,
    df_test['label'].values,
)


In [ ]:
benchmark_df = pd.DataFrame(benchmark_rows).sort_values('binary_f1', ascending=False)
print(benchmark_df)
benchmark_df.to_csv(artifact_path('xgboost_feature_set_benchmark.csv'), index=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=benchmark_df, x='feature_set', y='train_seconds', ax=axes[0])
axes[0].set_title('Training Time by Feature Set')
axes[0].set_xlabel('Feature set')
axes[0].set_ylabel('Seconds')
axes[0].tick_params(axis='x', rotation=20)

plot_df = benchmark_df.melt(
    id_vars=['feature_set'],
    value_vars=['hierarchical_f1', 'binary_f1', 'binary_precision', 'binary_recall'],
    var_name='metric',
    value_name='score',
)
sns.barplot(data=plot_df, x='feature_set', y='score', hue='metric', ax=axes[1])
axes[1].set_title('Benchmark Metrics by Feature Set')
axes[1].set_xlabel('Feature set')
axes[1].set_ylabel('Score')
axes[1].tick_params(axis='x', rotation=20)
axes[1].legend(title='metric')

plt.tight_layout()
plt.show()